# Methaniminium SHARC tutorial

Analyzes all 9 completed trajectories of methaniminium. Every trajectory starts on **S2** and was propagated for 100 fs with SA(4 singlets | 3 triplets)-CASSCF(6,4)/cc-pVDZ.

The file `methaniminium_tutorial_data.zip` contains the extracted SHARC results

## 1. Load the zip file containing the trajectory data

run this cell and upload `methaniminium_tutorial_data.zip` when prompted

In [ ]:
from pathlib import Path
import zipfile

DATA_DIR = Path("methaniminium_tutorial_data")
ARCHIVE = Path("methaniminium_tutorial_data.zip")

if not DATA_DIR.is_dir():
    if not ARCHIVE.is_file():
        try:
            from google.colab import files
            print("Please upload methaniminium_tutorial_data.zip")
            uploaded = files.upload()
            candidates = [name for name in uploaded if name.lower().endswith(".zip")]
            if not candidates:
                raise FileNotFoundError("No ZIP file was uploaded.")
            ARCHIVE = Path(candidates[0])
        except ImportError as exc:
            raise FileNotFoundError(
                "Place methaniminium_tutorial_data.zip beside the notebook."
            ) from exc
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as archive:
        archive.extractall(DATA_DIR)

trajectory_dirs = sorted(DATA_DIR.glob("TRAJ_*"))
print(f"Loaded {len(trajectory_dirs)} trajectories:")
print(", ".join(path.name for path in trajectory_dirs))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 120})

STATE_NAMES = ["S0", "S1", "S2", "S3", "T1", "T2", "T3"]
COMPONENT_TO_ROOT = np.array([0, 1, 2, 3, 4, 5, 6, 4, 5, 6, 4, 5, 6])

def load_xyz(path):
    lines = Path(path).read_text().splitlines()
    symbols, frames, pos = None, [], 0
    while pos < len(lines):
        if not lines[pos].strip():
            pos += 1
            continue
        n_atoms = int(lines[pos])
        block = lines[pos + 2:pos + 2 + n_atoms]
        if symbols is None:
            symbols = [line.split()[0] for line in block]
        frames.append([[float(x) for x in line.split()[1:4]] for line in block])
        pos += n_atoms + 2
    return symbols, np.asarray(frames)

def distance(xyz, a, b):
    return np.linalg.norm(xyz[:, a] - xyz[:, b], axis=1)

def dihedral(xyz, a, b, c, d):
    p0, p1, p2, p3 = (xyz[:, i] for i in (a, b, c, d))
    b0, b1, b2 = -(p1-p0), p2-p1, p3-p2
    b1 = b1 / np.linalg.norm(b1, axis=1)[:, None]
    v = b0 - (b0*b1).sum(axis=1)[:, None]*b1
    w = b2 - (b2*b1).sum(axis=1)[:, None]*b1
    return np.degrees(np.arctan2((np.cross(b1, v)*w).sum(axis=1),
                                 (v*w).sum(axis=1)))

## 2. Parsing all of the trajectories to read them easier


In [ ]:
records = {}
quality_rows = []

for folder in trajectory_dirs:
    name = folder.name
    energy = np.loadtxt(folder / "energy.out", comments="#")
    coeff = np.loadtxt(folder / "coeff_MCH.out", comments="#")
    listing = np.loadtxt(folder / "output.lis", comments="#")
    spin = np.loadtxt(folder / "spin.out", comments="#")
    fosc = np.loadtxt(folder / "fosc.out", comments="#")
    fosc_act = np.loadtxt(folder / "fosc_act.out", comments="#")
    symbols, xyz = load_xyz(folder / "output.xyz")

    amplitudes = coeff[:, 2:].reshape(len(coeff), 13, 2)
    component_pop = np.sum(amplitudes**2, axis=2)
    root_pop = np.column_stack([
        component_pop[:, 0], component_pop[:, 1],
        component_pop[:, 2], component_pop[:, 3],
        component_pop[:, [4, 7, 10]].sum(axis=1),
        component_pop[:, [5, 8, 11]].sum(axis=1),
        component_pop[:, [6, 9, 12]].sum(axis=1),
    ])
    active_component = listing[:, 3].astype(int) - 1
    active_root = COMPONENT_TO_ROOT[active_component]
    active_onehot = np.eye(7)[active_root]
    total = listing[:, 6]

    records[name] = dict(
        time=listing[:, 1], energy=energy, listing=listing, xyz=xyz,
        root_pop=root_pop, active_root=active_root,
        active_onehot=active_onehot, spin=spin, fosc=fosc, fosc_act=fosc_act,
    )
    quality_rows.append({
        "trajectory": name,
        "points": len(listing),
        "final_time_fs": listing[-1, 1],
        "energy_drift_eV": total[-1] - total[0],
        "energy_range_eV": np.ptp(total),
        "max_step_jump_eV": np.max(np.abs(np.diff(total))),
        "population_norm_error": np.max(np.abs(root_pop.sum(axis=1)-1)),
    })

quality = pd.DataFrame(quality_rows).set_index("trajectory")
print("Atom order:", symbols)
display(quality.style.format(precision=4))

## 4. Ensemble electronic populations

**Top Plot**

The quantum populations are shown (AKA the average MCH coefficient populations using the electronic wavefunction coefficients)

This plot describes how the propagated electronic wavefunction is distributed among the states


For each trajectory,

$$ P_i (t) = |c_i (t)|^2$$

where $$c_i$$ is the complex amplitude of electronic state i

A single trajectory simultaneously has $$ P_{S_0}(t), P_{S_1}(t), P_{S_2}(t)$$

We average these populations over the trajectories. The shaded region is the standard error of that mean. The three spin components of each triplet root are summed into T1, T2, and T3.


**Bottom Plot**

The classical populations are shown (fraction of trajectories on each active MCH root)

Each trajectory's nuclei move on only 1 active PES at any instant
A trajectory is therefore counted as: 1 on its active state and 0 on every other state

The population is then $$ P_i^{classical} (t) = \frac{\text{number of trajectories active on state i}}{\text{total # of trajectories }} $$

It is “classical” since it is obtained by counting the active surfaces that are followed by all of the classical nuclear trajectories


Between 0 and 30 fs, all trajectories changed from the initial S2 state through the S1 state to the S0 ground state. The triplet states remain completely unpopulated


**For large ensembles, the classical and quantum populations should look more similar**

In [ ]:
names = list(records)
time = records[names[0]]["time"]
quantum_pop = np.stack([records[n]["root_pop"] for n in names])
active_pop = np.stack([records[n]["active_onehot"] for n in names])

def plot_population_ensemble(values, title):
    mean = values.mean(axis=0)
    sem = values.std(axis=0, ddof=1) / np.sqrt(values.shape[0])
    fig, ax = plt.subplots()
    for i, label in enumerate(STATE_NAMES):
        line, = ax.plot(time, mean[:, i], label=label)
        ax.fill_between(time, mean[:, i]-sem[:, i], mean[:, i]+sem[:, i], color=line.get_color(), alpha=0.14)
    ax.set(xlabel="Time (fs)", ylabel="Population", ylim=(-0.04, 1.04),title=title)
    ax.legend(ncol=4)
    plt.grid(False)
    plt.show()
    return mean, sem

quantum_mean, quantum_sem = plot_population_ensemble(quantum_pop, "Quantum populations of the trajectories on each active MCH root (shading is SEM)")

active_mean, active_sem = plot_population_ensemble(active_pop, "Classical populations of the trajectories on each active MCH root")

##Time Evolution of C=N bond distance

The individual C=N bond distances can be generated for each trajectory along with the ensemble average (black) as a function of time. Photoexcitation rapidly weakens and elongates the C=N bond, most obvious during the S2 to S1 relaxation


The trajectories show an early C=N elongation
All trajectories begin near 1.30 Å, consistent with a C=N double bond

Between approximately 6.5 and 9 fs, most extend past 1.6 Å while moving from S2 toward S1

This is consistent with excited-state weakening of the π bond and increasing C=N torsion

In [ ]:
geometry_rows = []
cn_all = []
for name, rec in records.items():
    xyz = rec["xyz"]
    cn = distance(xyz, 0, 1)
    cn_all.append(cn)
    geometry_rows.extend({"trajectory": name, "time_fs": t, "C_N_A": r} for t, r in zip(time, cn))

cn_all = np.asarray(cn_all)
fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(names):
    ax.plot(time, cn_all[i], alpha=0.55, linewidth=0.9)
ax.plot(time, cn_all.mean(axis=0), color="black", linewidth=2.2,
        label="ensemble average")
ax.set(xlabel="Time (fs)", ylabel="C–N distance (Å)")
plt.grid(False)
ax.legend()
plt.show()

##State-change events

Below, a table of each trajectory's active root (what state it is populating) is shown at certain time intervals below

In [ ]:
event_rows = []
for name, rec in records.items():
    roots = rec["active_root"]
    changed = np.r_[True, roots[1:] != roots[:-1]]
    for step in np.flatnonzero(changed):
        event_rows.append({
            "trajectory": name, "step": step, "time_fs": rec["time"][step],
            "active_root": STATE_NAMES[roots[step]],
        })
events = pd.DataFrame(event_rows)
display(events)

##Simulated transient absorption spectrum

For the current active state, `fosc_act.out` contains 13 transition energies and their signed oscillator strengths. Downward transitions have negative oscillator strength and form the red ground-state-bleach/stimulated-emission contribution. Upward transitions have positive oscillator strength and form the blue excited-state-absorption (ESA) contribution.

The lines are broadened with a Gaussian of 1.0 eV FWHM on a 50-point energy grid.

**The spectrum is incomplete because only four singlets and three triplets were included. Could be complete with higher excited states**


In [ ]:
FWHM_EV = 1.0
N_ENERGY_POINTS = 50

all_transition_energies = np.concatenate([
    rec["fosc_act"][:, 1:14].ravel() for rec in records.values()
])
energy_grid = np.linspace(
    max(0.0, all_transition_energies.min() - FWHM_EV),
    all_transition_energies.max() + FWHM_EV,
    N_ENERGY_POINTS,
)

def gaussian_lines(centers, amplitudes):
    delta = energy_grid[None, None, :] - centers[:, :, None]
    kernel = np.exp(-4*np.log(2) * (delta/FWHM_EV)**2)
    return np.sum(amplitudes[:, :, None] * kernel, axis=1)

negative_by_traj = []
positive_by_traj = []
for name in names:
    active_spectrum = records[name]["fosc_act"]
    centers = active_spectrum[:, 1:14]
    strengths = active_spectrum[:, 14:27]
    negative_by_traj.append(gaussian_lines(centers, np.minimum(strengths, 0)))
    positive_by_traj.append(gaussian_lines(centers, np.maximum(strengths, 0)))

negative_ta = np.mean(negative_by_traj, axis=0)
positive_ta = np.mean(positive_by_traj, axis=0)
total_ta = negative_ta + positive_ta

limit = np.max(np.abs(total_ta))
fig, ax = plt.subplots(figsize=(10, 4))
mesh = ax.pcolormesh(time, energy_grid, total_ta.T, shading="auto", cmap="RdBu", vmin=-limit, vmax=limit)
ax.set(xlabel="Pump-probe delay (fs)", ylabel="Probe energy (eV)", title="simulated transient absorption spectrum")
fig.colorbar(mesh, ax=ax, label="Intensity")
plt.show()

Ground state bleech and excited-state
fluorescence (negative absorption) are shown in red, and excited-state absorption (positive absorption) is shown in blue.



*   There is a strong excited-state emission at early times, where many trajectories are in the excited state, with an energy that
decreases with time due to nuclear motion
*   At longer times, the trajectories are back in the ground state
and absorb in a broad energy range, because the molecules have a lot of energy and thus vibrate/dissociate.